# Task 3.2 — Failure Mode: Highly Overlapping Multi-class Data

In [1]:
# ── Imports & seed ────────────────────────────────────────────────
import os, numpy as np, random
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE); random.seed(RANDOM_STATE)

RESULTS_DIR = "/Users/abhijeetkrshah/Desktop/230036-midsem/partB/results"
os.makedirs(RESULTS_DIR, exist_ok=True)


## Failure Scenario Description

We generate a synthetic 3-class dataset with `class_sep=0.3` — far below the default of 1.0, creating **severe class overlap** in the 2-D feature plane.

This deliberately violates two assumptions from Task 1.2:

- **Assumption 1 (kernel/RKHS):** A linear kernel yields an RKHS identical to $\mathbb{R}^2$; with `class_sep=0.3`, no linear hyperplane can separate the three overlapping Gaussian clouds, so Definition 1's margin constraints $K_1 h_{y_i}(x_i)-h_k(x_i)\ge K_2-\xi_{ik}$ require enormous slack $\xi_{ik}$ for nearly every point.

- **Assumption 2 (valid QP / unique solution):** The QP still has a unique solution (the problem remains convex), but that solution is degenerate — the slack variables absorb the overlap, the margin collapses to near-zero, and the boundary reflects class centroids rather than a true discriminative structure.

This mirrors the paper's CB513 challenge (23–32% error) where protein secondary-structure classes partially overlap in sequence-feature space, motivating the use of profile kernels that map sequences to a more separable representation.

In [2]:
# ── Generate overlapping dataset ──────────────────────────────────
X, y = make_classification(
    n_samples=300, n_features=2, n_informative=2,
    n_redundant=0, n_classes=3, n_clusters_per_class=1,
    class_sep=0.3, random_state=RANDOM_STATE)
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
print(f"Train {X_train.shape}, Test {X_test.shape}")


Train (240, 2), Test (60, 2)


In [3]:
# ── Train LinearSVC (CS) and Random Forest ─────────────────────────
cs = LinearSVC(multi_class='crammer_singer', C=1.0,
               max_iter=5000, random_state=RANDOM_STATE)
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
cs.fit(X_train, y_train); rf.fit(X_train, y_train)

cs_err = (1 - accuracy_score(y_test, cs.predict(X_test))) * 100
rf_err = (1 - accuracy_score(y_test, rf.predict(X_test))) * 100
print(f"Crammer-Singer M-SVM  error: {cs_err:.2f}%")
print(f"Random Forest         error: {rf_err:.2f}%")


Crammer-Singer M-SVM  error: 46.67%
Random Forest         error: 16.67%


In [4]:
# ── Decision boundary plots ────────────────────────────────────────
def plot_boundary(ax, model, X, y, title):
    h = 0.04
    x0, x1 = X[:,0].min()-.5, X[:,0].max()+.5
    y0, y1 = X[:,1].min()-.5, X[:,1].max()+.5
    xx, yy = np.meshgrid(np.arange(x0,x1,h), np.arange(y0,y1,h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='Set1')
    ax.scatter(X[:,0], X[:,1], c=y, cmap='Set1', edgecolors='k', s=28, linewidth=0.4)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Feature 1"); ax.set_ylabel("Feature 2")

fig, axes = plt.subplots(1, 2, figsize=(13,5))
plot_boundary(axes[0], cs, X_train, y_train,
              f"Crammer-Singer M-SVM\nTest Error: {cs_err:.2f}%")
plot_boundary(axes[1], rf, X_train, y_train,
              f"Random Forest\nTest Error: {rf_err:.2f}%")
fig.suptitle("Failure Mode: class_sep=0.3 (Assumptions 1 & 2 violated)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
p = os.path.join(RESULTS_DIR, "failure_mode.png")
plt.savefig(p, dpi=150); plt.show(); print(f"Saved → {p}")


Saved → /Users/abhijeetkrshah/Desktop/230036-midsem/partB/results/failure_mode.png


/var/folders/t3/st41wyns54d8gvsrqb52wvgh0000gn/T/ipykernel_4877/3686935619.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(p, dpi=150); plt.show(); print(f"Saved → {p}")


## Analysis: Why the M-SVM Fails Here

The Crammer-Singer M-SVM fails because the core margin constraint in Definition 1 — $K_1 h_{y_i}(x_i)-h_k(x_i)\ge K_2-\xi_{ik}$ with $K_1=K_2=1$ — requires a **linear boundary** that scores the correct class above all others by at least 1 (minus slack).
When class distributions overlap as severely as `class_sep=0.3` induces in 2-D, every candidate hyperplane simultaneously violates this constraint for many training points, forcing $\xi_{ik}$ to absorb the entire overlapping region.
The dual solution converges to a boundary that splits the space based on global class centroids rather than local discriminative geometry, producing the blurry linear partition visible in the left plot.
Random Forest escapes this limitation by using axis-aligned recursive splits that can localise decision rules to small overlapping regions without assuming global linear separability.
The key lesson from the MSVMpack perspective is that all four models in Table 1 (WW, CS, LLW, MSVM2) are fundamentally **large-margin linear classifiers in their RKHS** — if the chosen kernel does not map the data to a representation where classes are approximately linearly separable, no amount of Frank-Wolfe optimisation recovers a good classifier.
This is why the CB513 experiments in the paper use profile-based string kernels: those kernels implicitly transform protein sequences into a space where secondary-structure classes are more separable, which is the real engineering contribution beyond the software unification.
The Frank-Wolfe stopping criterion $U(\alpha)$ can declare convergence even when the converged solution has high test error — dual optimality does not guarantee good generalisation if the model class is wrong for the data geometry.

**Suggested modification:** Using a non-linear RBF kernel SVM or the MSVM2 quadratic loss formulation (Table 1) may better handle overlapping boundaries by softening the penalty structure.
